In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
movies = pd.read_csv(
    "../data/processed/movies_processed.csv"
)

print("Movies loaded:", movies.shape)

Movies loaded: (3883, 5)


In [3]:
movies.head()

,movieId,title,genres,genres_clean,content
0,1,Toy Story (1995),Animation|Children's|Comedy,Animation Children's Comedy,Toy Story (1995) Animation Children's Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Adventure Children's Fantasy,Jumanji (1995) Adventure Children's Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance,Grumpier Old Men (1995) Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama,Comedy Drama,Waiting to Exhale (1995) Comedy Drama
4,5,Father of the Bride Part II (1995),Comedy,Comedy,Father of the Bride Part II (1995) Comedy


In [4]:
movies[["title", "genres", "content"]].head(10)

,title,genres,content
0,Toy Story (1995),Animation|Children's|Comedy,Toy Story (1995) Animation Children's Comedy
1,Jumanji (1995),Adventure|Children's|Fantasy,Jumanji (1995) Adventure Children's Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama,Waiting to Exhale (1995) Comedy Drama
4,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995) Comedy
5,Heat (1995),Action|Crime|Thriller,Heat (1995) Action Crime Thriller
6,Sabrina (1995),Comedy|Romance,Sabrina (1995) Comedy Romance
7,Tom and Huck (1995),Adventure|Children's,Tom and Huck (1995) Adventure Children's
8,Sudden Death (1995),Action,Sudden Death (1995) Action
9,GoldenEye (1995),Action|Adventure|Thriller,GoldenEye (1995) Action Adventure Thriller


In [5]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["content"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (3883, 4369)


In [6]:
movie_similarity = cosine_similarity(
    tfidf_matrix
)

print("Similarity matrix shape:", movie_similarity.shape)

Similarity matrix shape: (3883, 3883)


In [7]:
movie_indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

print("Movie lookup created successfully!")

Movie lookup created successfully!


In [8]:
movie_indices["Toy Story (1995)"]

np.int64(0)

In [9]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Similarity matrix shape:", movie_similarity.shape)

TF-IDF matrix shape: (3883, 4369)
Similarity matrix shape: (3883, 3883)


In [10]:
movie_indices["Toy Story (1995)"]

np.int64(0)

In [11]:
def recommend_similar_movies(movie_title, n=10):
    
    # Check if movie exists
    if movie_title not in movie_indices:
        return "Movie not found."
    
    # Get the index of the movie
    idx = movie_indices[movie_title]
    
    # Get similarity scores for this movie
    similarity_scores = list(
        enumerate(movie_similarity[idx])
    )
    
    # Sort movies by similarity score
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    # Remove the movie itself
    similarity_scores = similarity_scores[1:n+1]
    
    # Get movie indices
    movie_indices_list = [
        i[0] for i in similarity_scores
    ]
    
    # Return movie information
    recommendations = movies.iloc[
        movie_indices_list
    ][["title", "genres"]].copy()
    
    # Add similarity score
    recommendations["similarity_score"] = [
        i[1] for i in similarity_scores
    ]
    
    return recommendations

In [12]:
recommend_similar_movies("Toy Story (1995)", 10)

,title,genres,similarity_score
3045,Toy Story 2 (1999),Animation|Children's|Comedy,0.918706
3331,We're Back! A Dinosaur's Story (1993),Animation|Children's,0.451628
2892,"Story of Us, The (1999)",Comedy|Drama,0.422236
2039,L.A. Story (1991),Comedy|Romance,0.362859
12,Balto (1995),Animation|Children's,0.344689
2205,Lilian's Story (1995),Drama,0.334895
292,"Pyromaniac's Love Story, A (1995)",Comedy|Romance,0.323146
236,"Goofy Movie, A (1995)",Animation|Children's|Comedy|Romance,0.316685
47,Pocahontas (1995),Animation|Children's|Musical|Romance,0.308217
2490,"King and I, The (1999)",Animation|Children's,0.307286


In [13]:
recommend_similar_movies("Shawshank Redemption, The (1994)", 10)

,title,genres,similarity_score
382,S.F.W. (1994),Drama,0.324780
249,I.Q. (1994),Comedy|Romance,0.202560
286,Only You (1994),Comedy|Romance,0.202560
336,"War, The (1994)",Adventure|Drama,0.194398
278,Nobody's Fool (1994),Drama,0.155149
457,Go Fish (1994),Drama|Romance,0.148547
279,Nell (1994),Drama,0.147498
404,8 Seconds (1994),Drama,0.147498
468,I'll Do Anything (1994),Comedy|Drama,0.147402
378,Wolf (1994),Drama|Horror,0.146728


In [14]:
import pickle

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("../models/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("../models/movie_similarity.pkl", "wb") as f:
    pickle.dump(movie_similarity, f)

print("Content-based model files saved successfully!")

Content-based model files saved successfully!


In [15]:
recommend_similar_movies("This Movie Does Not Exist", 10)

'Movie not found.'